# TP7 – Introducción a Databricks y Apache Spark

## Descripción del proceso
En este trabajo cargamos el dataset Iris desde un archivo CSV almacenado en un
Volume de Databricks y lo leímos con Spark. Realizamos un proceso ETL básico con
PySpark, guardamos los datos transformados en formato Delta Lake (con control de
versiones) y generamos visualizaciones con matplotlib.

## Dataset
- Fuente: Iris Dataset (seaborn-data / UCI), descargado como CSV y subido a un Volume
- 150 registros, 5 columnas: sepal_length, sepal_width, petal_length, petal_width, species

## Resultados principales
- Hay 3 especies con 50 muestras cada una antes de limpiar. Al eliminar un duplicado
  exacto quedan 149 filas y virginica pasa a tener 49.
- La especie virginica tiene los pétalos más largos (promedio de unos 5,56 cm).
- El ratio largo/ancho del pétalo separa muy bien a setosa (unos 6,9) de las otras dos;
  versicolor (3,2) y virginica (2,8) quedan más cerca entre sí.
- Delta Lake guarda cada versión de la tabla: pudimos borrar filas, leer la versión
  original y restaurarla.

Cargar el CSV con Spark desde un Volume de Databricks

In [0]:
df = ( spark.read.option("header",True).option("inferSchema", True).csv("/Volumes/workspace/default/datos/iris.csv") )

df.show(5)


Ver el esquema (tipos de datos)

In [0]:
df.printSchema()

Estadisticas Basicas

In [0]:
df.describe().show()

Consultas con Spark SQL

In [0]:
df.createOrReplaceTempView("iris")

spark.sql("""
  SELECT species, COUNT(*) as cantidad
  FROM iris
  GROUP BY species
""").show()

Eliminar duplicados y manejar nulos

In [0]:
df_clean = df.dropDuplicates()
df_clean = df_clean.dropna() # Elimina filas con valores nulos.

print(f"Filas originales: {df.count()}")
print(f"Filas limpias: {df_clean.count()}")

In [0]:
# Filas repetidas en el dataset original
df.groupBy(df.columns).count().filter("count > 1").show()

Crear columna calculada (ratio largo/ancho de pétalo)

In [0]:
from pyspark.sql.functions import col, round as spark_round

df_transform = df_clean.withColumn(
    "petal_ratio",
    spark_round(col("petal_length") / col("petal_width"), 2)
)

df_transform.show(5)

Filtrar solo la especie "virginica"

In [0]:
df_virginica = df_transform.filter(col("species") == "virginica")
df_virginica.show()

Agregacion - Promedio de medidas por especie

In [0]:
from pyspark.sql.functions import avg

df_transform.groupBy("species").agg(
    avg("sepal_length").alias("avg_sepal_length"),
    avg("petal_length").alias("avg_petal_length"),
    avg("petal_ratio").alias("avg_petal_ratio")
).show()

Join: unimos el DataFrame transformado con una tabla auxiliar de especies (código y color) usando la columna species

In [0]:

#Es un join interno: conserva solo las filas que tienen coincidencia en ambas tablas. Las 149 filas se mantienen porque todas las especies existen en la tabla auxiliar.

# Tabla auxiliar: un código y un color para cada especie
info_especies = spark.createDataFrame(
    [("setosa", 1, "steelblue"), ("versicolor", 2, "salmon"), ("virginica", 3, "seagreen")],
    ["species", "species_id", "color"]
)

# Join interno por especie: agrega species_id y color a cada flor
df_join = df_transform.join(info_especies, on="species", how="inner")
df_join.show(5)
print(f"Filas tras el join: {df_join.count()}")  # tiene que dar 149: no se pierde ninguna fila

Guardar el DataFrame transformado en formato Delta

In [0]:
# Guardamos como tabla Delta administrada en el catálogo
df_transform.write.format("delta").mode("overwrite").saveAsTable("iris_transformada")

# Leer de vuelta
df_delta = spark.table("iris_transformada")
df_delta.show(5)
print(f"Filas leídas: {df_delta.count()}")

In [0]:
# Verificar que el schema sea correcto
df_delta.printSchema()

# Verificar que no haya nulos
df_delta.describe().show()

In [0]:
# Detalle de la tabla: formato y cantidad de archivos
spark.sql("DESCRIBE DETAIL iris_transformada").show(truncate=False)

Delta Lake guarda cada versión de la tabla. Borramos algunas filas, leemos la versión original con VERSION AS OF 0 y luego restauramos la tabla.

In [0]:
# Borramos algunas filas para generar una versión nueva
spark.sql("DELETE FROM iris_transformada WHERE petal_length > 6.5")
spark.sql("SELECT COUNT(*) AS filas_actuales FROM iris_transformada").show()

# Leemos la versión 0 (antes del borrado): tienen que ser 149 filas
spark.sql("SELECT COUNT(*) AS filas_version_0 FROM iris_transformada VERSION AS OF 0").show()

# Historial de cambios de la tabla
spark.sql("DESCRIBE HISTORY iris_transformada").select("version", "operation").show()

In [0]:
# Restauramos la tabla a su versión original
spark.sql("RESTORE TABLE iris_transformada TO VERSION AS OF 0")
spark.sql("SELECT COUNT(*) AS filas_restauradas FROM iris_transformada").show()

Grafico de barras - cantidad de flores por especie

Convertir a pandas para graficar

In [0]:
import matplotlib.pyplot as plt 

pdf_conteo = df_transform.groupBy("species").count().toPandas()

plt.figure(figsize=(8, 4))
plt.bar(pdf_conteo["species"], pdf_conteo["count"], color=["steelblue", "salmon", "seagreen"])
plt.title("Cantidad de flores por especie")
plt.xlabel("Especie")
plt.ylabel("Cantidad")
plt.tight_layout()
plt.show()

Grafico de lineas - largo de petalo por indice (ordenado)

In [0]:
pdf_sorted = df_transform.orderBy("petal_length").toPandas().reset_index(drop=True)

plt.figure(figsize=(10, 4))
for especie, grupo in pdf_sorted.groupby("species"):
    plt.plot(grupo.index, grupo["petal_length"], label=especie)

plt.title("Largo de pétalo por especie (ordenado)")
plt.xlabel("Índice")
plt.ylabel("Petal Length (cm)")
plt.legend()
plt.tight_layout()
plt.show()

## Conclusiones
- El gráfico de barras confirma que las especies están balanceadas: 50 muestras cada
  una, salvo virginica con 49 tras eliminar un duplicado.
- En el gráfico de líneas, setosa tiene los pétalos más cortos (alrededor de 1,5 cm),
  versicolor queda en el medio (4,3 cm) y virginica tiene los más largos (5,6 cm).
- El largo del pétalo, solo o combinado con el ancho, permite distinguir las especies;
  setosa se separa por completo de las otras dos.
- Delta Lake permitió volver a una versión anterior de la tabla después de borrar filas.
- Databricks nos dio Spark, Delta y el catálogo de datos ya configurados, sin
  administrar el cómputo.